In [ ]:
%pip install elasticsearch==8.19.0
%restart_python

### Patch awards-v4 with `_full` denormalized siblings + re-sync (oxjob #123.2)

After `BuildAwardsV4.ipynb` set up the rich nested versions (`institution_awarded`, `primary_topic`, `topics`), this notebook:

1. PUTs new mappings on awards-v4 for the `_full` denormalized siblings (`institution_awarded_full`, `primary_topic_full`, `topics_full`) — object type, explicit `keyword` on id fields.
2. Full-syncs `openalex.awards.awards_api` → awards-v4 (the `_full` columns are populated by the updated `CreateAwardsAPI.ipynb`; sync_awards-style push handles them via `_source.*`).

**Prereqs:**
- ALTER awards_api to add the 3 `_full` columns has been run.
- Updated `CreateAwardsAPI.ipynb` (oxjob #123.2 commit) has been run so awards_api has the `_full` columns populated.
- BuildAwardsV4 has been run (awards-v4 exists with the rich nested versions).

**After this:** verify v4 with direct ES term queries on `*_full` fields, then merge `oxjob-123.2-awards-v4-cutover` branch in openalex-elastic-api to deploy.

In [ ]:
import json
from pyspark.sql import functions as F
from elasticsearch import Elasticsearch, helpers
import logging

logging.basicConfig(level=logging.WARNING, format='[%(asctime)s]: %(message)s')
log = logging.getLogger(__name__)

ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")

CONFIG = {
    "table_name": "openalex.awards.awards_api",
    "target_index": "awards-v4",
}

client = Elasticsearch(
    hosts=[ELASTIC_URL],
    max_retries=3,
    request_timeout=180,
)

assert client.indices.exists(index=CONFIG['target_index']), \
    f"{CONFIG['target_index']} not found; run BuildAwardsV4.ipynb first"
print(f"OK: {CONFIG['target_index']} exists")

In [ ]:
# Topic _full sub-structure: same shape as the nested topic_props, but as object
topic_full_props = {
    "id":           {"type": "keyword"},
    "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    "score":        {"type": "float"},
    "subfield": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
    "field": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
    "domain": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
}

patch = {
    "properties": {
        "institution_awarded_full": {
            # NOT nested — denormalized object for fast term filtering by elastic-api
            "properties": {
                "id":           {"type": "keyword"},
                "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                "ror":          {"type": "keyword"},
                "country_code": {"type": "keyword"},
                "type":         {"type": "keyword"},
                "lineage":      {"type": "keyword"},
            },
        },
        "primary_topic_full": {"properties": topic_full_props},
        "topics_full":        {"properties": topic_full_props},
    }
}

print("PUT mapping:")
print(json.dumps(patch, indent=2))

client.indices.put_mapping(index=CONFIG['target_index'], body=patch)
print(f"Mapping added to {CONFIG['target_index']}")

# Verify the new fields are present in the mapping
mapping = client.indices.get_mapping(index=CONFIG['target_index'])[CONFIG['target_index']]['mappings']['properties']
for field in ('institution_awarded_full', 'primary_topic_full', 'topics_full'):
    assert field in mapping, f"{field} missing from v4 mapping after PUT"
print("All 3 _full fields confirmed in v4 mapping")

In [ ]:
# Full re-sync awards_api -> awards-v4. The _full columns (populated by the
# updated CreateAwardsAPI run) flow through automatically via _source = STRUCT(*).
def send_partition_to_elastic(partition, index_name):
    client = Elasticsearch(
        hosts=[ELASTIC_URL],
        max_retries=3,
        request_timeout=180,
    )

    def generate_actions():
        for row in partition:
            yield {
                "_op_type": "index",
                "_index": index_name,
                "_id": row.id,
                "_source": row._source.asDict(True),
            }

    count = 0
    for success, info in helpers.parallel_bulk(
        client, generate_actions(),
        chunk_size=500, thread_count=4,
    ):
        count += 1
        if not success:
            print(f"FAILED TO INDEX: {info}")
            raise Exception(f"Failed to index document: {info}")
    print(f"Indexed {count} docs to {index_name}")


df = spark.read.table(CONFIG['table_name'])
df = (df
    .withColumn("id", F.concat(F.lit("https://openalex.org/G"), F.col("id")))
    .select("id", F.struct(F.col("*")).alias("_source"))
)
df = df.repartition(96)
print(f"Total rows to sync: {df.count()}")

target = CONFIG['target_index']
df.foreachPartition(lambda p: send_partition_to_elastic(p, target))
print("Re-sync complete")

In [ ]:
# Verify _full filtering works without .keyword and without nested query wrapping
client.indices.refresh(index=CONFIG['target_index'])

# Topic filter via _full (mirrors what elastic-api will do post-cutover)
r1 = client.search(index=CONFIG['target_index'], body={
    "size": 0,
    "query": {"term": {"primary_topic_full.id": "https://openalex.org/T11668"}},
})
print(f"primary_topic_full.id = T11668: {r1['hits']['total']['value']}")

# Compare to the existing v3 .keyword workaround value (sanity check)
v3_compare = client.search(index="awards-v3", body={
    "size": 0,
    "query": {"term": {"primary_topic.id.keyword": "https://openalex.org/T11668"}},
})
print(f"v3 primary_topic.id.keyword = T11668: {v3_compare['hits']['total']['value']}")
print("==> these should match")

# institution_awarded_full filter (no nested needed)
r2 = client.search(index=CONFIG['target_index'], body={
    "size": 0,
    "query": {"term": {"institution_awarded_full.country_code": "CH"}},
})
print(f"institution_awarded_full.country_code = CH: {r2['hits']['total']['value']}")

# Sample doc to eyeball the _full structure
sample = client.search(index=CONFIG['target_index'], body={
    "size": 1,
    "query": {"exists": {"field": "institution_awarded_full"}},
    "_source": ["institution_awarded", "institution_awarded_full", "primary_topic_full"],
})
print("Sample doc with _full fields:")
print(json.dumps(sample['hits']['hits'][0]['_source'], indent=2)[:1500])